# Anomaly Detection Prototype

Goal: detect unusual behavior in existing feature or risk series before building a production anomaly detector.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import numpy as np
import pandas as pd

from ai_models.risk_detector import run_systemic_risk_detector

risk = run_systemic_risk_detector(
    prices_path="data/prices_cache.parquet",
    treasury_path="data/treasury_yields_cache.parquet",
    benchmark_ticker="SPY",
)
risk.tail()

In [ ]:
risk = risk.copy()
risk["RiskZScore"] = (
    (risk["RiskScore"] - risk["RiskScore"].rolling(60, min_periods=20).mean())
    / risk["RiskScore"].rolling(60, min_periods=20).std()
)
risk["IsAnomaly"] = risk["RiskZScore"].abs() >= 2.5
risk[risk["IsAnomaly"]].tail(20)

In [ ]:
# Optional if scikit-learn is installed.
try:
    from sklearn.ensemble import IsolationForest

    X = risk[["RiskScore"]].dropna()
    clf = IsolationForest(random_state=42, contamination=0.05)
    pred = clf.fit_predict(X)
    detected = X.copy()
    detected["AnomalyFlag"] = pred
    display(detected[detected["AnomalyFlag"] == -1].tail(20))
except ModuleNotFoundError:
    print("Install scikit-learn to test IsolationForest-based anomaly detection.")